# Spark Exact Quantile Analysis

Proposed paper title: "A Quick and Exact Method for Distributed Quantile Computation"

In this document we include a time complexity analysis of an exact quantile algorithm designed for
Spark.

```
```


<center> Table Notation </center>

| **Symbol**        | **Definition**                                                                               |
|-------------------|----------------------------------------------------------------------------------------------|
| $n$               | Total number of elements across all partitions.                                              |
| $n_i$             | Total elements in the $i$th partition.                                                       |
| $P$               | Number of partitions.                                                                        |
| $q$               | Specific quantile level being queried (e.g., 0.5 for median).                                |
| $\Delta k$        | Rank distance between approximate and exact quantile; satisfies $|\Delta k| \leq \varepsilon n$. |
| $\pi$             | Pivot value.                                                                                  |
| $\varepsilon$     | Desired approximation error; quantile estimates are within $\varepsilon n$ rank positions.   |
| $C_i$             | Number of elements in the $i$th partition that are strictly less than the pivot.             |
| $\tau$            | latency per communication.                                                                   |
| $g$               | time to transmit a communication                                                             |
| $L$               | latency per superstep (or stage in Spark parlance)                                           |

We assume

$$P_{\text{part}} = c\,P_{\text{core}}, \qquad c\ge 1$$

Thus we can use $P$ to denote both $P_{\text{part}}$ and $P_{\text{core}}$ without
affecting the time or space complexity analyses.

## Note on cross-references.

`@ id` marks a definition and `# id` refers to it.  If you install the jupyterlab-mdx jupyterlab
extension <https://github.com/dosirrah/jupyterlab-mdx>, labels and references will be assigned
appropriate numbers every time you run a markdown cell.


## Computation Models

<center>Table Comparison between Computation Models </center>

|  Feature      |  PRAM          | BSP                                  | CGM                |Spark                                       |
|---------------|----------------|--------------------------------------|--------------------|--------------------------------------------|
| Processors    | up to $O(n)$   | no assumption                        | no assumption      | $P \ll n$                                  |
| Memory        | Shared, global | Local, partitioned                   | Local, partitioned | Local, partitioned                         |
| Access        | Instant        | Local = instant, remote $h + L$      | Local = instant, remote = many-to-many | Local = instant, remote = shuffle, network |
| Communication | $O(1)$ shared memory access | $g h + L$, Expensive    | Expensive          | Expensive                                  |
| Sync          | Lockstep       | Superstep, $L$                       | Round              | Stage (often at shuffle), $L$              |

### Parallel Random-Access Machine (PRAM)

Early work on parallel selection targets the PRAM model.  Notable examples include Cole's
parallel selection ^Cole88 and Han’s deterministic selection ^Han07.  These algorithms assume
shared memory with $O(1)$ time per shared memory access.   Spark, by contrast, is a shared-nothing
platform: data is partitioned, each partition is distributed to an executor (JVM) and memory
accesses within a JVM are assumed to be fast, and communication between executors is orders
of magnitude slower.

Because PRAM abstracts away network communication under the assumption of $O(1)$ shared
memory access, algorithms designed for PRAM would not translate well to the Spark model.

We therefore cite PRAM results for historical context but do not evaluate them further.

### Bulk-Synchronous Parallel (BSP)

A BSP machine has $P$ processors each with local memory. Execution proceeds in *supersteps*.
Each superstep is comprised of

- Local computation
- Point-to-point communication
- Barrier synchronization

The output from one superstep becomes the input to the next.

The time per superstep is modeled as described in ^Valiant90,

$$T = \max(\text{local work}) + g \cdot h + L$$

where $h$ is the maximum communication in words sent by a processor,
$P$ is processors, $g$ is inverse bandwidth, $L$ superstep boundary
latency.

- $h = \max_{1\le i\le P}$
$\bigl(\text{num words processor }i\text{ sends or receives in that superstep}\bigr)$.
    * It is measured in words (or bytes if you multiply by word-size).
	* Only the largest send/receive total among all processors matters, because the superstep cannot finish until that heaviest sender/receiver is done.


The notion of superstep and stage are similar. A superstep comprises the following

1. Local computation on private memory.
2. Arbitrary point-to-point communications.
3. Global barrier: all processes complete a set of operations and synchronize, all messages delivered.

A *stage* is a contiguous set of transformations that can be pipelined without a shuffle
(i.e., only narrow dependencies inside).  A shuffle, broadcast join, or a cached boundary
breaks the pipeline (i.e., a wide dependency) and starts a new stage.

### Coarse-Grained Multiprocessor (CGM)

CGM and BSP are similar.  CGM is a theoretical model that allows worst-case
analysis while abstracting communication cost into a single many-to-many
communication between all processors in each round.  BSP is a more
detailed model that explicitly accounts for computation, latency and bandwidth
in each superstep.

We mention CGM because it is used by Al-Furaih^AlFuraih97 in his description
of Randomized Select.

In this analysis we adapt algorithms for the Spark computational model.


### Spark Model

Spark ^Zahari2010_Spark ^Zaharia2012_RDD ^Zaharia2016_Apache was introduced as a practical system
that unifies multiple paradigms.  It introduces the notion of a Resilient Distributed Dataset^Zaharia2012_RDD.
Each RDD is comprised of a set of partitions distributed across executor nodes.  A partition resides in
a single executor's memory where possible and spills onto disk only if necessary.  The architecture
is a "shared-nothing" architecture in that there is no shared memory between executors.  As with BSP,
computation proceeds locally in each processor (executor) and then when necessary synchronization
between nodes occurs, but the similarities are superficial.   Spark introduces multiple concepts
that do not exist in the BSP model including RDDs, narrow and wide transformations, lazy evaluation,
and actions.

RDDs are themselves immutable and instead Spark applies a transformation to one immutable RDD to
create another immutable RDD.  Transformations that operate on a single partition are known as
narrow transformations.  Transformations that involve moving data between partitions (e.g.,
shuffles) are considered wide transformations.   Narrow transformations affect each
partition in an RDD independently and thus such transformations can proceed asynchronously in
parallel.  Because wide transformations involve communicating between partitions, they necessarily
introduce dependencies.  Only when all dependencies have been satisfied can a computation continue.

Spark introduces the notion of an *action*.  The Spark user specifies transformations on RDDs.
Each transformation is represented as an edge in a Directed Acyclic Graph (DAG).  This DAG
constitutes a lineage: a kind of plan.  The plan is not executed immediately: Spark uses
*lazy evaluation*.   The Spark user builds the DAG by specifying the set of transformations
they wish to perform on their RDDs.  Only when the Spark user requests a result does the plan
execute.  The call to obtain a result is called an *action*.

Spark also introduces the notion of a *stage*.  Spark stages are analogous to BSP supersteps.   Both
represent a barrier before computation can proceed.  However, Spark's notion adds significant
complexity.  Within a stage maximal chains of narrow transformations proceed in parallel
until a wide transformation blocks further progress until that wide transformation completes.

When Spark encounters a wide transformation anywhere in the lineage of the action, it closes out
the entire current stage. No tasks in the subsequent stage—across all branches of the DAG including
those unaffected by the wide transformation—will start until that wide transformation is fully
complete and then the global barrier is crossed.

Spark’s lazy evaluation and DAG scheduler introduce potential optimizations not present in classic BSP.

**Assumption:**

The number of data partitions $P$ is at most a constant factor larger
than the number of executor cores available. Under this assumption, the
number of parallel waves is constant, and thus the wall-clock runtime
complexity remains asymptotically equivalent to the per-wave complexity.
Here wave refers to a set of concurrently processed partitions.

## Performance Evaluation Procedure

Except for historical Sample Sort, which we never implemented
in the Spark context, we will characterize Spark select algorithm
performance along the following dimensions:

1.  Executor time
2.  Driver time
3.	Executor memory
4.  Driver memory
5.  Work
6.  Network latency (barriers + RPC/tree-reduces)
7.	Network volumes (bytes moved cluster-wide)
8.	Number of Shuffles
9.	Number of Actions
10.  Number of Persists

Executor time complexity refers to the cost on the critical path--
the longest dependency chain that dictates wall-clock runtime.

Executor work complexity refers to the total computation performed
by the cluster averaged over the executors. Time complexity
therefore captures the *depth* of the computation, while work
complexity captures the *total effort*.

For the same total work, the job with long dependency chains or
deep trees exhibit longer runtimes than a job that allows all
work to proceed in parallel.

For a driver, there is only one for any given job and thus
the time and work complexity are identical.

Whether one minimizes average per-executor work or time depends on
the situation.  If a cluster is running many jobs, minimizing the
work performed by the cluster will ensure better usage of resources.
If you are only interested in minimizing the time to execute this
one job, time is more important.

## Sequential External-Memory Sample Sort

Frazer and McKellar's Samplesort ^Frazer70 is a sequential
sorting algorithm that is designed for a single processor that sorts
$n$ items where $n$ is much larger than the available memory.  It
selects a value $m$ such that $n/m$ can fit in memory.  The implementor
samples $m$ from the $n$ using an unspecified method.  These $m$ are
assumed to be drawn randomly from the $n$.  The node sorts the $m$ and
then uses each as a bound of $m+1$ buckets.  The system then reads the
data into memory a few at a time (as a stream).  For each element it
finds which bucket it falls in and then writes it into a file for
that bucket.  Once all the data has been written to the $m+1$ files,
they can each be read sequentially into memory, sorted and written back
out.  The concatenation of the $m+1$ buckets is now sorted.

1. Memory‐budgeted sampling:
  Choose $m$ so that $n/m$ (a bucket size) fits in RAM, then draw an
  (unspecified) random sample of $m$ keys from the $n$.

2. Splitter construction:
  Sort the $m$ samples in memory and use them as splitters for $m+1$
  output buckets.

4. Single‐pass bucketing:
  Stream the $n$ items: for each key, determine its bucket by comparing
  against the m splitters, and append it to that bucket’s file (on disk).

5. Recursive in‐memory sorts:
  Each bucket now has at most $\lceil n/(m+1)\rceil$ items; load each into
  RAM, sort it (e.g. Quicksort), and write it out.

6.	Concatenation:
  Finally, concatenate the sorted bucket outputs to produce the globally
  sorted sequence.

## Parallel Sorting by Regular Sampling (PSRS)

Shi, H. & Schaeffer, J. (1992). Parallel Sorting by Regular
Sampling. Journal of Parallel and Distributed Computing, 14(4), 361–372.

Shi et al ^Shi92_PSRS extends Frazer et al's Sample Sort, such
that it is parallel and defines a method for
sampling.  Shi et al's extension is often called "Parallel Sample Sort"
or simply "Sample Sort"; the distinction between Shi et al. and
Frazer et al. is usually clear from context.

Shi et al's Sample Sort assumes a MIMD processor.  However,
it will work with $P$ separate processors that each have only local memory.

precondition: data $X$ is distributed among $P$ processors such
that each processor processes $w=n/P$ elements.

Phase 1: each of $P$ processors sorts its local block of $w$ data using sequential
Quicksort.  When all $P$ processors are done, $X$ is said to be
locally ordered.

Phase 2: each processor generates a *regular sample* $Y$.

A *regular sample* is defined as

$$X_{\tfrac{w}{P} + jw}, X_{\tfrac{2w}{P} + jw}, \dots, X_{\tfrac{(P-1)w}{P} + jw}$$

where $0 \leq j \leq P-1$.  $j$ is a particular processor.  These are the
$\frac{w}{P}$-th, $\frac{2w}{P}$-th, … elements of processor $j$’s local sorted block

Each processor creates a sample of size $P-1$ and communicates these
$P-1$ samples to a single node which sorts the regular sample $Y$
using a sequential Quicksort.  Pick $P-1$ samples from the $Y$ as the
pivots (a.k.a., splitters) for partitioning $X$.  The samples are
picked evenly from $Y$, i.e.,

$$Y_{\tfrac{P}{2}}, Y_{\tfrac{P}{2} + P}, \dots, Y_{\tfrac{P}{2} + P(P-2)}$$

The central node broadcasts the splitters $$Y$$ to the processors.
Each processor uses binary search to find smallest element $\geq$ to each
pivot (splitter).  This breaks up each processor's data into $P$
sorted sublists.

Phase 3:  Each processor $i$ keeps its $i$th sublist and passes the
others to the appropriate processor.  Each processor $i$ performs a
$P$-way merge on the $i$th sublists of $P$ lists.

After sorting the processors synchronize, the sort is done.


### PSRS BSP-style Time complexity

This is a BSP-style time complexity analysis.  While looking at papers related to Spark, I found
that people typically do not do a BSP-style time complexity analysis, but rather break out the
various costs into separate metrics as described in Section #pe.

- Phase 1:

  $$O(w \log w) = O\left(\tfrac{n}{P} \log \tfrac{n}{P}\right)$$

- Phase 2: the central processor sorts the regular sample using Quicksort.
  Picks the $P-1$ samples.  Then each processor performs $P-1$ binary searches
  to find the splitters.

  $$O \left(P^2 \log P^2 + (P-1) + (P-1) \log \tfrac{n}{P}\right)$$

  which simplifies to

  $$O \left(P^2 \log P^2 + (P-1) \log \tfrac{n}{P}\right)$$


- Phase 3: each node passes $P-1$ sublists to the other processors.  The original
  paper does not discuss the overhead of communicating the sublists.  In BSP
  terminology this would add a cost:

  $$O(g\tfrac{n}{P})$$

  In distributed dataflow systems like Spark, this maps to shuffle read/write bytes
  and spill cost, which can dominate for large $P$.

  Once the sublists have been distributed, each processor performs a merge of
  $P$ sublists, each of length $\tfrac{w}{P}$.  The merge is accomplished
  recursively as in mergesort resulting in $\log P$ merges.

  $$O \left(\tfrac{n}{P^2} P \log P \right) = O \left(\tfrac{n}{P} \log P \right)$$

The expected total cost thus becomes

$$O\left(\tfrac{n}{P} \log \tfrac{n}{P} + P^2 \log P^2 + (P-1) \log \tfrac{n}{P} + g\tfrac{n}{P} + \tfrac{n}{P} \log P  \right) \tag{@eq:ss}$$

If $n > P^3$ then Equation #eq:ss becomes

$$T_{SS} = O \left(\tfrac{n}{P} \log \tfrac{n}{P} + g\tfrac{n}{P} \right) \tag{@eq:Tss}$$

If we drop the BSP notation and separate the performance characterization into categories of cost.
The time complexity for the processors (executors) becomes

$$T_{exec} = O \left(\tfrac{n}{P} \log \tfrac{n}{P} \right)$$

## Spark Sort

The Spark sort is sometimes called "range-partition-then-sort."  It is similar to
PSRS, but it doesn't sort each partition up front.

1. Sampling (without full local sort)

Each executor samples a constant $m$ records per partition.

2. Collect

The samples are sent to the driver via a `collect()`. The driver collects
from each executor the $m$ records per partition.

3. Driver sorts and picks splitters.

The driver picks $P-1$ split points at the associated quantiles of the sample.

4. Driver broadcasts splitters.

The driver broadcasts the split points via a `TorrentBroadcast`.

5. Range-partitioning shuffle

For each partition, every record is assigned a partition ID by performing
a binary search on the splitters.  The result is $P$ unordered sublists
where all values in sublist $i$ are smaller than all in sublist $i+1$.
For each partition, the executor sends the $i$th sublist to the $i$th
new partition.

6. Local sort within each partition.

Each executor runs Spark's `UnsafeExternalSorter` on its partition resulting from
the range-partitioning shuffle.  As records arrive, they are placed in
a buffer.  If the partition fits in a single buffer then it is sorted
entirely in-memory using a dual-pivot Quicksort.   If a partition reaches the
spill threshold then it is broken into buffers and each buffer as it arrives is
sorted using a dual-pivot Quicksort on primitive arrays and written to disk.
Once all buffers created from a partition are written, the executor performs a
$k$-way merge where $k$ here is the number of buffers comprising the partition.

6. Local sort within each partition.

As buckets arrive, they are buffered.  If a partition fits in a single buffer without
reaching a "spill threshold" then the executor for that partition performs a dual-pivot Quicksort
on the entire partition in memory.

$$T_{localsort} = O(n_i \log n_i) = O\left(\tfrac{n}{P} \log \tfrac{n}{P}\right)$$

If a partition fills past a threshold, Spark uses an external sort.  Let $m$ denote the number
records that fit in a buffer before the threshold is reached.  When the threshold is reached,
after sorting, the buffer is "spilled" to disk.  This results in
$b_i = \lceil \tfrac{n_i}{m}\rceil$ buffers.

$$T_{localsort} = b \cdot O( m \log m) = O( n_i \log m)$$

Once all $b_i$ buffers have been sorted, Spark Sort performs a sequential $b_i$-way merge.  This is
accomplished by constructing a min-heap bounded to size $b_i$ containing the smallest element
of each buffer.  The smallest item starts at the top of the first min-heap.  After popping
the minimum, it is streamed to disk, and the next smallest from that buffer is pushed to
the min-heap.   This continues until all items have been written to disk.

$$T_{localmerge} = O(n_i \log b_i)$$

$$T_{local} = T_{localsort} + T_{localmerge}$$

$$T_{local} = O( n_i \log m + n_i \log b_i) = O( n_i \log m + n_i \log \tfrac{n_i}{m}) = O(n_i \log n_i)$$

$$T_{local} = O\left(\tfrac{n}{P} \log \tfrac{n}{P} \right)   \tag{@eq:spsort_local}$$

Combining #eq:spsort_sample and #eq:spsort_collect - #eq:spsort_local yields the expected time of

$$T = T_{sample} + T_{collect} + T_{splitters} + T_{broadcast} + T_{shuffle} + T_{local}$$

$$T =
O(\tfrac{n}{P}) +
(g k + O(P k) + L) +
(O\left(P k \log P k\right)) +
(O\left( g P \log P + L \right)) +
(O(\tfrac{n}{P} \log P + g \tfrac{n}{P} + L)) +
(O\left(\tfrac{n}{P} \log \tfrac{n}{P}\right)) \tag{@eq:spsort_total1}$$

Assuming $n > P^2 k$, #eq:spsort_total1 and expressing in big-$O$ notation yields
a worst-case bound on expected time of

$$T =
O\left(\tfrac{n}{P} \log \tfrac{n}{P} + (g k + P k) + g P \log P + g \tfrac{n}{P} + L\right)$$

which simplifies to a BSP-style time complexity of

$$T = O\left(\tfrac{n}{P} \log \tfrac{n}{P} + g P \log P + g \tfrac{n}{P} + L\right)\tag{@eq:spsort_total2}$$
